In [47]:
# -----------------------------------------------------------
# A.0 – Load the raw Food.com recipes file
# -----------------------------------------------------------
# This cell loads the RAW_recipes.csv file into a pandas DataFrame.
# The dataset will be cleaned and transformed in later steps.

import ast
from collections import Counter, defaultdict
import pandas as pd

# Path to the raw Food.com recipes file within the project structure
RAW_RECIPES_PATH = "../data/raw/RAW_recipes.csv"

# Load the CSV file into a pandas DataFrame called 'df'.
# A DataFrame works like a spreadsheet: it has rows (recipes) and columns (fields).
df = pd.read_csv(RAW_RECIPES_PATH)

# Quick sanity check: show basic info about the dataset
print(f"Loaded RAW_recipes from {RAW_RECIPES_PATH}")
print(f"Number of rows (recipes): {len(df)}")
print(f"Number of columns: {df.shape[1]}")

# Display the first 5 rows of the dataset (for interactive inspection in Jupyter)
df.head()


Loaded RAW_recipes from ../data/raw/RAW_recipes.csv
Number of rows (recipes): 231637
Number of columns: 12


,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,arriba baked winter squash mexican style,137739,55,47892,2005-09-16,"['60-minutes-or-less', 'time-to-make', 'course...","[51.5, 0.0, 13.0, 0.0, 2.0, 0.0, 4.0]",11,"['make a choice and proceed with recipe', 'dep...",autumn is my favorite time of year to cook! th...,"['winter squash', 'mexican seasoning', 'mixed ...",7
1,a bit different breakfast pizza,31490,30,26278,2002-06-17,"['30-minutes-or-less', 'time-to-make', 'course...","[173.4, 18.0, 0.0, 17.0, 22.0, 35.0, 1.0]",9,"['preheat oven to 425 degrees f', 'press dough...",this recipe calls for the crust to be prebaked...,"['prepared pizza crust', 'sausage patty', 'egg...",6
2,all in the kitchen chili,112140,130,196586,2005-02-25,"['time-to-make', 'course', 'preparation', 'mai...","[269.8, 22.0, 32.0, 48.0, 39.0, 27.0, 5.0]",6,"['brown ground beef in large pot', 'add choppe...",this modified version of 'mom's' chili was a h...,"['ground beef', 'yellow onions', 'diced tomato...",13
3,alouette potatoes,59389,45,68585,2003-04-14,"['60-minutes-or-less', 'time-to-make', 'course...","[368.1, 17.0, 10.0, 2.0, 14.0, 8.0, 20.0]",11,['place potatoes in a large pot of lightly sal...,"this is a super easy, great tasting, make ahea...","['spreadable cheese with garlic and herbs', 'n...",11
4,amish tomato ketchup for canning,44061,190,41706,2002-10-25,"['weeknight', 'time-to-make', 'course', 'main-...","[352.9, 1.0, 337.0, 23.0, 3.0, 0.0, 28.0]",5,['mix all ingredients& boil for 2 1 / 2 hours ...,my dh's amish mother raised him on this recipe...,"['tomato juice', 'apple cider vinegar', 'sugar...",8


### A.1 – Converting ingredient strings to lists

Here I turn the raw ingredient strings into real Python lists using a safe parser.  
I also keep a backup of the original column for debugging.


In [48]:
# Keep the original raw ingredient strings for debugging and reporting
df["ingredients_raw"] = df["ingredients"]

# Count how many ingredient strings fail to parse
parse_failures = 0

def safe_literal_eval(x):
    """
    Safely convert a string representation of a Python list into an actual list.
    If parsing fails, return an empty list and count the failure.
    """
    global parse_failures

    try:
        return ast.literal_eval(x)
    except Exception:
        parse_failures += 1
        return []

# Convert each row's ingredient string into a list object
df["ingredients"] = df["ingredients"].apply(safe_literal_eval)

# Print a single summary message if any failures occurred
if parse_failures > 0:
    print(f"Ingredient list parsing failures: {parse_failures}")

# Preview a few rows to verify the conversion worked
df["ingredients"].head()


0    [winter squash, mexican seasoning, mixed spice...
1    [prepared pizza crust, sausage patty, eggs, mi...
2    [ground beef, yellow onions, diced tomatoes, t...
3    [spreadable cheese with garlic and herbs, new ...
4    [tomato juice, apple cider vinegar, sugar, sal...
Name: ingredients, dtype: object

### A.2 – Lowercasing ingredients

This step standardises all ingredient text by converting everything to lowercase. Even though it was observed that everything was lowercase, this step was needed to make sure everything works correctly. It prevents duplicate entries caused by casing differences and keeps the data consistent.


In [49]:
def lowercase_ingredient_list(ingredients_list):
    """
    Convert all ingredient strings in a list to lowercase.
    Includes safety checks so the pipeline does not break
    if unexpected values appear.
    """
    if not isinstance(ingredients_list, list):
        return []

    cleaned = []
    for ing in ingredients_list:
        if isinstance(ing, str):
            cleaned.append(ing.lower())
        else:
            # Non-string ingredients (rare, but safe to handle)
            cleaned.append(str(ing).lower())

    return cleaned

# Apply the lowercase transformation to every ingredient list
df["ingredients"] = df["ingredients"].apply(lowercase_ingredient_list)

# Quick preview to confirm success
df["ingredients"].head()


0    [winter squash, mexican seasoning, mixed spice...
1    [prepared pizza crust, sausage patty, eggs, mi...
2    [ground beef, yellow onions, diced tomatoes, t...
3    [spreadable cheese with garlic and herbs, new ...
4    [tomato juice, apple cider vinegar, sugar, sal...
Name: ingredients, dtype: object

### A.3 – Token frequency exploration (diagnostic)

This is an exploratory step to understand which words appear most often in ingredient strings. I use it to spot descriptor-like tokens (“fresh”, “chopped”, ...) and refine my cleaning rules. It isn’t part of the main pipeline but helps justify later design decisions.

In [50]:
import re

# Show more rows when printing DataFrames
pd.set_option("display.max_rows", 200)

token_counts = Counter()

# Tokenise each ingredient into alphabetic tokens only
for ingredient_list in df["ingredients"]:
    for ing in ingredient_list:
        # Extract alphabetic tokens (filters out %, numbers, punctuation)
        tokens = re.findall(r"[A-Za-z]+", ing)
        for t in tokens:
            token_counts[t] += 1

# Convert to DataFrame
token_df = (
    pd.DataFrame(
        {
            "token": list(token_counts.keys()),
            "count": list(token_counts.values()),
        }
    )
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)

print("Top 100 most frequent alphabetic tokens:")
display(token_df.head(100))


Top 100 most frequent alphabetic tokens:


,token,count
0,salt,126481
1,pepper,119077
2,sugar,94253
3,oil,80876
4,cheese,76540
5,fresh,75568
6,garlic,72372
7,butter,71551
8,onion,65240
9,ground,65009


### Step 1 – Removing descriptor words

This was done by writing all the common adjectives that could be detected and then removing them. 
Here I strip out adjectives like “fresh”, “chopped”, “unsalted”, ...  
These words describe preparation, not ingredient identity, so removing them gives cleaner, more consistent ingredient names.
"fresh chopped tomatoes" -> "tomatoes"
"unsalted butter"        -> "butter"

In [51]:
# A dataset-derived list of descriptor words that describe size, preparation,
# state, salt/fat content, or general adjectives — NOT ingredient identity.
GENERIC_DESCRIPTORS = {
    # state / processing
    "fresh", "freshly", "frozen", "canned", "condensed", "evaporated", "prepared",
    "instant", "quick-cooking", "dried", "dry", "raw", "cooked", "smoked",
    "pickled", "refrigerated", "split",

    # cut / form
    "ground", "chopped", "diced", "minced", "sliced", "crushed",
    "shredded", "grated", "whole", "chunky", "pieces", "nuggets",
    "fillets", "steaks", "cubes", "crumbs", "crumb", "rings",

    # fat / sugar / sodium labels
    "low-fat", "reduced-fat", "fat-free", "nonfat", "light", "lite",
    "skim", "salt-free", "sugar-free", "reduced-calorie",

    # salt / butter / sweetness descriptors
    "unsalted", "salted", "sweet", "sweetened", "unsweetened",
    "dark",

    # meat / fish trim
    "lean", "extra", "boneless", "skinless", "center", "country-style", "baby",

    # size / texture / count
    "small", "medium", "large", "miniature", "little", "thin", "thick",
    "chunk", "granular", "granulated", "coarse", "fine",
    "head", "heads", "stem", "stems", "spears",
    "wedge", "wedges", "halves", "half",

    # leaves / herbs physical bits
    "leaf", "leaves", "sprig", "sprigs",
    "rind", "zest", "segments",

    # temperature
    "cold", "hot", "warm", "iced",

    # flour / cereal descriptors
    "self-rising", "self-raising", "whole wheat", "whole grain",
    "unbleached", "white", "brown", "multigrain",

    # misc adjectives
    "creamy", "plain", "sharp", "mild", "soft", "hard", "quick", "powdered",
    "ripe",

    # colour descriptors
    "black", "green", "yellow",

    # cooking states / forms
    "baking", "breasts",

    # shape/form
    "chips", "flakes",

    # generic modifiers
    "mix", "seasoning",

    # other generic / non-identity modifiers
    "bell", "sour", "heavy", "virgin", "breast", "cooking", "paste", "stock", "seeds", "italian",

    "boiling", "semi-sweet", "all-purpose", "jack", "sea", "spray", "dressing", "mapple", "cool", "half-and-half", 
}

# Multi-word descriptor phrases that should be removed as a unit
GENERIC_DESCRIPTOR_PHRASES = {
    "low sodium",
    "reduced sodium",
    "fat free",
    "sugar free",
    "reduced fat",
    "non fat",
    "whole wheat",
    "whole grain",
    "self rising",
    "self raising",
    "dark brown",
    "light brown",
}


def remove_descriptors_from_ingredient(ingredient: str) -> str:
    """
    Remove descriptor words and phrases from a single ingredient string.

    We assume ingredients are already lowercased earlier in the pipeline.
    If everything is removed, we fall back to the original ingredient
    to avoid producing an empty string.
    """
    original = ingredient.strip()
    ing = original  # already lowercase from previous step

    # Remove multi-word descriptor phrases first
    for phrase in GENERIC_DESCRIPTOR_PHRASES:
        ing = ing.replace(phrase + " ", " ")
        ing = ing.replace(" " + phrase, " ")
        if ing == phrase:
            ing = ""

    # Split into words and remove single-word descriptors
    words = ing.split()
    cleaned_words = [w for w in words if w not in GENERIC_DESCRIPTORS]

    if len(cleaned_words) == 0:
        # fallback: return original ingredient if we stripped everything
        return original

    return " ".join(cleaned_words)


def remove_descriptors_from_list(ingredients_list):
    """
    Apply descriptor removal to every ingredient in the list.
    """
    return [remove_descriptors_from_ingredient(ing) for ing in ingredients_list]


# Apply descriptor cleaning to the dataset
df["ingredients"] = df["ingredients"].apply(remove_descriptors_from_list)

# Inspect the cleaned version
df["ingredients"].head()


0    [winter squash, mexican, mixed spice, honey, b...
1    [pizza crust, sausage patty, eggs, milk, salt ...
2    [beef, onions, tomatoes, tomato, tomato soup, ...
3    [spreadable cheese with garlic and herbs, new ...
4    [tomato juice, apple cider vinegar, sugar, sal...
Name: ingredients, dtype: object

### Step 2 – Plural → singular normalisation

Here I convert plural ingredient words into their singular form to reduce duplication  
(e.g., “tomatoes” and “tomato” becoming the same token).  
The logic is simple and safe:  
- handle a few irregular forms (ingredients only),  
- apply lightweight rules (“ies”→“y”, “oes”→drop “es”, final “s”→remove when appropriate),  
- avoid changing words like “bass” or “couscous”.  
This keeps ingredient identities consistent without over-stemming.

In [52]:
import string  # needed for stripping punctuation like commas

# Irregular ingredient forms that do NOT follow the normal "just remove s/es" rules.
# These are ingredient words we expect to appear in the dataset.
IRREGULAR_PLURAL_INGREDIENTS = {
    # truly irregular food words where the singular is different
    "loaves": "loaf",
    "leaves": "leaf",
    "halves": "half",

    # ingredients whose common plural is what we want to keep as-is
    # (otherwise "chives" would become "chive", "greens" would become "green")
    "chives": "chives",
    "greens": "greens",
}


def singularise_token(token):
    """
    Convert a single word to its singular form using minimal,
    safe plural rules that are appropriate for recipe ingredients.

    Parameters
    ----------
    token : str
        A single word from an ingredient string, for example "tomatoes" or "berries".

    Returns
    -------
    singular : str
        The singular form of the word (for example "tomato" or "berry"),
        or the original word if no rule applies.
    """
    # Clean up any extra spaces and ensure we are working in lowercase
    word = token.strip().lower()

    # Strip basic punctuation from the ends (e.g. "lemons," -> "lemons")
    word = word.strip(string.punctuation)

    # If the word is empty or contains non-alphabetic characters (e.g. "1/2"),
    # we simply return it unchanged.
    if not word or not word.isalpha():
        return word

    # -------------------------------------------------------
    # 1. Handle ingredient-specific irregular plurals first
    # -------------------------------------------------------
    if word in IRREGULAR_PLURAL_INGREDIENTS:
        return IRREGULAR_PLURAL_INGREDIENTS[word]

    # -------------------------------------------------------
    # 2. "berries" -> "berry", "cranberries" -> "cranberry"
    # -------------------------------------------------------
    if word.endswith("ies") and len(word) > 3:
        # Remove the "ies" and add "y"
        return word[:-3] + "y"

    # -------------------------------------------------------
    # 3. "tomatoes" -> "tomato", "potatoes" -> "potato"
    # -------------------------------------------------------
    if word.endswith("oes") and len(word) > 3:
        # Remove only the "es" at the end
        # "tomatoes" -> "tomatoe" (remove last 2 letters) -> "tomato"
        return word[:-2]

    # -------------------------------------------------------
    # 4. Generic plural rule: remove trailing 's'
    # -------------------------------------------------------
    # We want to catch words like "eggs" -> "egg", "carrots" -> "carrot".
    # But we want to avoid changing words where 's' is part of the base word,
    # such as "bass" or "couscous" ("us" ending).
    if word.endswith("s"):
        ends_with_double_s = word.endswith("ss")
        ends_with_us = word.endswith("us")

        # Only remove the final "s" if the word does not end in "ss" or "us"
        if not ends_with_double_s and not ends_with_us:
            return word[:-1]

    # -------------------------------------------------------
    # 5. If none of the rules applied, return the word as-is
    # -------------------------------------------------------
    return word


def normalise_plurals_in_ingredient(ingredient):
    """
    Apply plural → singular normalisation to each word inside
    a single ingredient string.

    Example:
        "fresh chopped tomatoes" -> "fresh chopped tomato"
        "potatoes"               -> "potato"
    """
    # Split the ingredient into individual words based on spaces
    tokens = ingredient.split()

    # Create a list that will store the singular versions
    singular_tokens = []

    # Go through each word one by one
    for token in tokens:
        singular_form = singularise_token(token)
        singular_tokens.append(singular_form)

    # Join the singular words back into a single string
    normalised_ingredient = " ".join(singular_tokens)

    return normalised_ingredient


def normalise_plurals_in_list(ingredients_list):
    """
    Apply plural normalisation to each ingredient in a list of ingredients.

    Parameters
    ----------
    ingredients_list : list of str
        A list of ingredient strings for a single recipe.

    Returns
    -------
    normalised_list : list of str
        A new list of ingredient strings where each ingredient has been
        processed by normalise_plurals_in_ingredient.
    """
    normalised_list = []

    for ingredient in ingredients_list:
        normalised_ingredient = normalise_plurals_in_ingredient(ingredient)
        normalised_list.append(normalised_ingredient)

    return normalised_list


# -----------------------------------------------------------
# Apply plural/singular normalisation to the DataFrame column
# -----------------------------------------------------------
df["ingredients"] = df["ingredients"].apply(normalise_plurals_in_list)

# Quick preview to visually confirm the transformation
df["ingredients"].head()


0    [winter squash, mexican, mixed spice, honey, b...
1    [pizza crust, sausage patty, egg, milk, salt a...
2    [beef, onion, tomato, tomato, tomato soup, rot...
3    [spreadable cheese with garlic and herb, new p...
4    [tomato juice, apple cider vinegar, sugar, sal...
Name: ingredients, dtype: object

### Step 3 – Filter out rare, overly common, and background “noise” ingredients

Here I remove ingredients that don’t contribute meaningful signal.  
I first printed a full frequency table of all ingredient names and used that  
to choose sensible thresholds:  
- **Too rare (< 20)** → not statistically reliable for trigger detection.  
- **Too common (> 60,000)** → appear in almost every recipe, so they provide no discrimination.  
- **Background noise terms** like *water, salt, oil* are removed regardless of frequency.

This step keeps only ingredients that are informative enough to act as meaningful triggers  
or meaningful similarity features.

In [53]:
# 3.1 Build a global frequency table of ingredients

# This list will contain every ingredient from every recipe.
all_ingredients = []

# Go through each recipe's ingredient list in the DataFrame
for ingredient_list in df["ingredients"]:
    # ingredient_list is something like: ["onion", "olive oil", "salt"]
    for ingredient in ingredient_list:
        all_ingredients.append(ingredient)

# Count how many times each ingredient appears across all recipes
ingredient_counts = Counter(all_ingredients)

# Optional: build a DataFrame with frequencies, sorted from most common
ingredient_frequency_df = (
    pd.DataFrame(
        {
            "ingredient": list(ingredient_counts.keys()),
            "count": list(ingredient_counts.values()),
        }
    )
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)

# You can inspect the top rows if you want:
# ingredient_frequency_df.head()


# 3.2 Define frequency thresholds
# These values are chosen so that:
#   - very rare ingredients (fewer than 20 occurrences) are removed,
#   - extremely common ingredients (more than 60,000 occurrences)
#     are also removed.
#
# You can adjust these later if needed, but this is a reasonable
# starting point for the Food.com dataset.

MIN_INGREDIENT_FREQUENCY = 20      # too rare  -> remove
MAX_INGREDIENT_FREQUENCY = 60000   # too common -> remove


# 3.3 Define a small set of "always remove" background ingredients
# These are ingredients that we do NOT want to treat as important,
# even if their frequency is within the allowed range.
#
# They are so generic and widespread that they add noise to:
#   - trigger detection
#   - similarity comparisons between recipes
#
# We can extend this list later if needed.

ALWAYS_REMOVE_INGREDIENTS = {
    "water",
    "salt",
    "salt and pepper",
    "oil",
    "olive oil",
    "vegetable oil",
    "ice",
}


# 3.4 Function to filter ingredients in a single recipe

def filter_by_frequency_and_background(ingredients_list,
                                       frequency_lookup,
                                       min_freq,
                                       max_freq,
                                       always_remove_set):
    """
    Remove ingredients from a single recipe that are:
      - too rare,
      - too common, or
      - explicitly listed as background "noise" ingredients.

    Parameters
    ----------
    ingredients_list : list of str
        The ingredients for one recipe, e.g. ["onion", "olive oil", "salt"].

    frequency_lookup : dict-like
        A mapping from ingredient string -> global count
        (we use 'ingredient_counts' for this).

    min_freq : int
        Minimum allowed frequency (ingredients below this are removed).

    max_freq : int
        Maximum allowed frequency (ingredients above this are removed).

    always_remove_set : set of str
        Ingredients that should always be removed, regardless of frequency.

    Returns
    -------
    filtered_ingredients : list of str
        The cleaned ingredient list for this recipe.
    """
    filtered_ingredients = []

    for ingredient in ingredients_list:
        # 1. Remove ingredients that are explicitly marked
        #    as background noise.
        if ingredient in always_remove_set:
            # Skip this ingredient
            continue

        # 2. Look up the global frequency for this ingredient
        ingredient_count = frequency_lookup.get(ingredient, 0)

        # 3. Remove ingredients outside the allowed range
        if ingredient_count < min_freq:
            # Too rare -> skip
            continue

        if ingredient_count > max_freq:
            # Too common -> skip
            continue

        # 4. If it passes all checks, keep the ingredient
        filtered_ingredients.append(ingredient)

    return filtered_ingredients


# 3.5 Apply filtering to the entire DataFrame

df["ingredients"] = df["ingredients"].apply(
    lambda recipe_ingredients: filter_by_frequency_and_background(
        recipe_ingredients,
        ingredient_counts,
        MIN_INGREDIENT_FREQUENCY,
        MAX_INGREDIENT_FREQUENCY,
        ALWAYS_REMOVE_INGREDIENTS,
    )
)

# Quick preview to make sure it worked
df["ingredients"].head()


0         [winter squash, mexican, mixed spice, honey]
1                     [pizza crust, egg, milk, cheese]
2    [beef, tomato, tomato, tomato soup, rotel toma...
3    [spreadable cheese with garlic and herb, new p...
4         [tomato juice, apple cider vinegar, mustard]
Name: ingredients, dtype: object

### Step 4 – Deduplicate ingredients and build the final vocabulary

At this stage all major cleaning is done, so now I ensure each recipe’s ingredient list is **unique and consistent**.  
I remove duplicate entries (e.g. `"egg", "egg"`) and sort them alphabetically.  
This gives each recipe a clean ingredient *set*, which is essential for building co-occurrence graphs later.

After deduplication, I recompute global ingredient frequencies one final time and build the **ingredient vocabulary table**.  
This assigns every ingredient a stable `ingredient_id`, which we will use in all downstream modelling steps.


In [54]:
# 4.1 Deduplicate and sort ingredients inside each recipe

def deduplicate_and_sort_ingredients(ingredients_list):
    """
    Take a list of ingredient strings for a single recipe and:
      - remove duplicates, and
      - sort them alphabetically.

    Example:
        ["egg", "onion", "egg"] -> ["egg", "onion"]
    """
    # Convert the list to a set to remove duplicates,
    # then back to a list and sort it for consistency.
    unique_ingredients = sorted(set(ingredients_list))

    return unique_ingredients


# Apply this to every recipe in the DataFrame
df["ingredients"] = df["ingredients"].apply(deduplicate_and_sort_ingredients)

# Quick visual check
df["ingredients"].head()


# 4.2 Recompute global ingredient frequencies (final version)
# Now that we have deduplicated and filtered ingredient lists,
# we build the final frequency table. You can choose one of two
# interpretations for "count":
#
#   A) Total occurrences across recipes
#      - If an ingredient appears twice in the same recipe, it counts twice.
#      - (In our case, duplicates are removed, so this matches recipe count.)
#
#   B) Number of recipes that contain the ingredient
#      - Each recipe contributes at most 1 to the count for that ingredient.
#
# Because we already deduplicated per recipe, both interpretations become
# the same. We simply count how many times each ingredient appears in the
# final cleaned lists.

final_all_ingredients = []

for ingredient_list in df["ingredients"]:
    for ingredient in ingredient_list:
        final_all_ingredients.append(ingredient)

final_ingredient_counts = Counter(final_all_ingredients)


# 4.3 Build the final ingredient vocabulary DataFrame
# We now turn the counts into a vocabulary table that assigns a
# numeric ID to each ingredient. This is helpful for:
#   - building matrices,
#   - storing embeddings,
#   - efficient lookup later in the project.

# First, create a DataFrame with ingredient and count
ingredient_vocab_df = (
    pd.DataFrame(
        {
            "ingredient": list(final_ingredient_counts.keys()),
            "count": list(final_ingredient_counts.values()),
        }
    )
    # Sort ingredients alphabetically so IDs are stable and readable
    .sort_values(by="ingredient")
    .reset_index(drop=True)
)

# Add a numeric ID column (0, 1, 2, ..., N-1)
ingredient_vocab_df["ingredient_id"] = ingredient_vocab_df.index

# Optional: reorder columns to have the ID first
ingredient_vocab_df = ingredient_vocab_df[["ingredient_id", "ingredient", "count"]]

# Take a look at the final vocabulary
ingredient_vocab_df.head()


# -----------------------------------------------------------
# 4.4 (Optional) Save cleaned recipes and vocabulary to disk
# -----------------------------------------------------------
# These paths are just examples; adjust them to match your project structure.

# Save the cleaned recipes (with final ingredient lists)
# df.to_csv("../data/processed/recipes_cleaned.csv", index=False)

# Save the ingredient vocabulary
# ingredient_vocab_df.to_csv("../data/processed/ingredient_vocabulary.csv", index=False)


,ingredient_id,ingredient,count
0,0,salsa,87
1,1,1 fat buttermilk,24
2,2,1 fat cottage cheese,32
3,3,1 milk,472
4,4,10-inch flour tortilla,48


### Step 5 – Remove recipes with too few ingredients

After cleaning, some recipes end up with very few usable ingredients.  
Recipes with fewer than 3 ingredients don’t provide meaningful co-occurrence information for graph-based modelling, so here I filter them out.  
This keeps only recipes that have enough structure for trigger detection and similarity analysis.


In [55]:
# We keep recipes that have at least 3 ingredients after cleaning.
MIN_INGREDIENTS_PER_RECIPE = 3

# Create a helper column that stores the length of each cleaned ingredient list
df["n_clean_ingredients"] = df["ingredients"].apply(len)

# Filter the DataFrame to keep only recipes with enough ingredients
df = df[df["n_clean_ingredients"] >= MIN_INGREDIENTS_PER_RECIPE].reset_index(drop=True)

# (Optional) You can drop the helper column if you don't need it later
# df = df.drop(columns=["n_clean_ingredients"])

# Quick check to confirm the filter worked
df["ingredients"].head()


0         [honey, mexican, mixed spice, winter squash]
1                     [cheese, egg, milk, pizza crust]
2    [beef, cheddar cheese, chili powder, cumin, ki...
3    [new potato, parsley, red pepper, red wine vin...
4         [apple cider vinegar, mustard, tomato juice]
Name: ingredients, dtype: object

### Step 6 – Data-driven handling of compound “X and Y” ingredients

Some ingredients in Food.com appear as combined phrases like **“salt and pepper”**, **“chips & salsa”**, or **“ham ’n cheese”**. These strings can either represent *one meaningful combined ingredient* or *two separate ingredients*.  
To avoid incorrect splitting (or incorrect merging), this step applies a **data-driven rule** that looks at actual ingredient frequencies in the dataset.

**Key logic I implemented:**
- If no connector (“and”, “&”, “’n”) appears → **leave ingredient unchanged**.  
- If the connector appears at the start/end (e.g., “and milk”, “celery &”) → treat it as a formatting issue and **keep the meaningful side only**.  
- If both sides exist:
  - Compare **frequency of the combined phrase** vs. **frequency of each side individually**.
  - If the combined phrase is **more common than both sides**, treat it as a *single ingredient*.
  - Otherwise, we **split it** — but only keep the sides that are frequent enough to be meaningful (within our global min/max thresholds).
  - If nothing satisfies these conditions, fall back to keeping the phrase as-is.

This allows the model to correctly distinguish true pairs (e.g., “mac and cheese”) from ingredients that should be separated for co-occurrence learning.  
After splitting, I deduplicate and rebuild the vocabulary again so that all future steps operate on the corrected ingredient list.


In [56]:
# The connector tokens we treat as "and" equivalents.
CONNECTOR_TOKENS = {"and", "&", "'n"}

# 6.1 Recompute ingredient frequencies on the current cleaned dataset
compound_all_ingredients = []

for ingredient_list in df["ingredients"]:
    for ingredient in ingredient_list:
        compound_all_ingredients.append(ingredient)

compound_ingredient_counts = Counter(compound_all_ingredients)


# 6.2 Helper to decide how to treat a compound ingredient
def split_compound_ingredient(
    ingredient,
    frequency_lookup,
    min_freq,
    max_freq,
):
    """
    Apply the 7-step logic to a single ingredient that may contain a connector.

    Cases:
      - No connector → [original ingredient].
      - Connector with NO word before it (e.g. "& salsa"):
        → drop connector, keep the other side exactly as-is.
      - Connector with words on both sides:
          1) Get counts for left, right, and combined phrase.
          2) If combined_count > left_count AND combined_count > right_count:
                keep [combined phrase].
          3) Otherwise:
                For each side where:
                    side_count > combined_count AND
                    min_freq <= side_count <= max_freq,
                keep that side as its own ingredient.
                If at least one side qualifies, return only those sides.
                If none qualify, fall back to [combined phrase].
    """
    text = ingredient.strip()
    if not text:
        return []

    tokens = text.split()

    # 1–2: Find the first connector, if any
    connector_index = None
    for i, token in enumerate(tokens):
        if token in CONNECTOR_TOKENS:
            connector_index = i
            break

    # If there is no connector at all → do nothing
    if connector_index is None:
        return [text]

    # Split into left / right around the connector
    left_tokens = tokens[:connector_index]
    right_tokens = tokens[connector_index + 1 :]

    left_part = " ".join(left_tokens).strip()
    right_part = " ".join(right_tokens).strip()

    # Case A: no word before the connector
    #   e.g. "& salsa", "and milk"
    #   → drop connector, keep the non-empty side exactly
    if not left_part and right_part:
        return [right_part]

    # Symmetric odd case: "celery &" → keep "celery"
    if left_part and not right_part:
        return [left_part]

    # If both sides are empty (ingredient was just "and" / "&" / "'n"),
    # drop it completely.
    if not left_part and not right_part:
        return []

    # Case B: both sides exist → apply data-driven logic
    combined_count = frequency_lookup.get(text, 0)
    left_count = frequency_lookup.get(left_part, 0)
    right_count = frequency_lookup.get(right_part, 0)

    # 6. If combined version is bigger than both sides → keep combined
    if combined_count > left_count and combined_count > right_count:
        return [text]

    # 7. Otherwise, check each side individually:
    #    - side_count > combined_count
    #    - min_freq <= side_count <= max_freq
    parts_to_keep = []

    if (
        left_count > combined_count
        and (left_count >= min_freq)
        and (left_count <= max_freq)
    ):
        parts_to_keep.append(left_part)

    if (
        right_count > combined_count
        and (right_count >= min_freq)
        and (right_count <= max_freq)
    ):
        parts_to_keep.append(right_part)

    # If at least one side qualifies, we drop the connector and keep only those.
    if parts_to_keep:
        return parts_to_keep

    # Fallback: no side qualified by thresholds → keep original phrase
    return [text]


# 6.3 Apply compound handling to every recipe
def expand_compound_ingredients_in_list(
    ingredients_list,
    frequency_lookup,
    min_freq,
    max_freq,
):
    """
    Apply the compound-splitting logic to all ingredients in a single recipe.
    """
    expanded_ingredients = []

    for ingredient in ingredients_list:
        split_or_keep = split_compound_ingredient(
            ingredient,
            frequency_lookup,
            min_freq=min_freq,
            max_freq=max_freq,
        )

        for item in split_or_keep:
            if item:  # ignore empty strings just in case
                expanded_ingredients.append(item)

    return expanded_ingredients


df["ingredients"] = df["ingredients"].apply(
    lambda ingredient_list: expand_compound_ingredients_in_list(
        ingredient_list,
        compound_ingredient_counts,
        MIN_INGREDIENT_FREQUENCY,
        MAX_INGREDIENT_FREQUENCY,
    )
)

# Quick check
df["ingredients"].head()


# 6.4 Deduplicate and sort again after compound handling
def deduplicate_and_sort_ingredients(ingredients_list):
    """
    Remove duplicate ingredients in a recipe and sort alphabetically.
    """
    return sorted(set(ingredients_list))


df["ingredients"] = df["ingredients"].apply(deduplicate_and_sort_ingredients)

df["ingredients"].head()


# 6.5 Rebuild the final ingredient vocabulary AFTER all cleaning
final_all_ingredients = []
for ingredient_list in df["ingredients"]:
    for ingredient in ingredient_list:
        final_all_ingredients.append(ingredient)

final_ingredient_counts = Counter(final_all_ingredients)

ingredient_vocab_df = (
    pd.DataFrame(
        {
            "ingredient": list(final_ingredient_counts.keys()),
            "count": list(final_ingredient_counts.values()),
        }
    )
    .sort_values(by="ingredient")
    .reset_index(drop=True)
)

ingredient_vocab_df["ingredient_id"] = ingredient_vocab_df.index
ingredient_vocab_df = ingredient_vocab_df[["ingredient_id", "ingredient", "count"]]

# Inspect a few rows to confirm
ingredient_vocab_df.head()


,ingredient_id,ingredient,count
0,0,1 fat buttermilk,24
1,1,1 fat cottage cheese,31
2,2,1 milk,460
3,3,10-inch flour tortilla,48
4,4,18 table cream,23


### Token Frequency Check (After All Cleaning Steps)

Here I quickly re-compute token frequencies on the *fully cleaned* ingredient lists.  
This is not part of the main pipeline — it’s just a diagnostic tool to:

- verify that our descriptor removal, plural normalisation, and compound splitting worked,
- spot any remaining noisy words that may need to be added to stop-lists,
- understand which ingredient tokens dominate the dataset after cleaning.

It helps validate that the processed data now contains meaningful, well-structured ingredient terms.


In [57]:
# Force Jupyter to show more rows
pd.set_option("display.max_rows", 200)

token_counts = Counter()

# Go through every ingredient string and split into tokens
for ingredient_list in df["ingredients"]:
    for ing in ingredient_list:
        for token in ing.split():
            token_counts[token] += 1

# Convert to DataFrame
token_df = (
    pd.DataFrame(
        {
            "token": list(token_counts.keys()),
            "count": list(token_counts.values()),
        }
    )
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)

print("Top 100 most frequent tokens:")
display(token_df.head(150))


Top 100 most frequent tokens:


,token,count
0,cheese,74201
1,garlic,70541
2,egg,62993
3,flour,57077
4,powder,50857
5,cream,49534
6,sauce,44344
7,juice,43469
8,chicken,42636
9,clove,41455


In [58]:
# # -----------------------------------------------------------
# # Test the compound-splitting behaviour
# # -----------------------------------------------------------

# for test in ["& salsa", "chips & salsa", "ham 'n cheese"]:
#     print(
#         test, "->",
#         split_compound_ingredient(
#             test,
#             compound_ingredient_counts,
#             MIN_INGREDIENT_FREQUENCY,   # min_freq
#             MAX_INGREDIENT_FREQUENCY    # max_freq
#             # min_part_frequency will use its default (50)
#         )
#     )

# # -----------------------------------------------------------
# # Display a large portion of the ingredient vocabulary
# # -----------------------------------------------------------

# # How many ingredients do you want to see?
# N = 2000   # you can increase to 3000, 5000, or even len(ingredient_vocab_df)

# # Increase pandas' row display limit
# pd.set_option("display.max_rows", N)

# # Sort the vocabulary alphabetically (should already be sorted, but just in case)
# sorted_vocab = ingredient_vocab_df.sort_values(by="ingredient").reset_index(drop=True)

# print(f"\nShowing the first {N} ingredients:\n")
# display(sorted_vocab.head(N))


## Grouping the Ingredients 

1- Split the ingredient into tokens/words.

2- Remove tokens that are not alphabetic or appear in the stop token list.

3- Among the remaining candidates, pick the token that appears most frequently in the whole dataset.

4- This uses global statistics to choose the most representative word (e.g. “egg” appears far more than “yolk”, so “egg” becomes the head).

5- If no valid token exists, the ingredient becomes its own group.

This was a repetitive process where the groups were ran over and over again. Each time the groups that should not be named as a group were banned from being a group name. Then the code was ran again. 

In [59]:
# Words that are allowed to stay in ingredient names,
# but should NEVER be used as group labels.
GROUPING_STOP_TOKENS = {
    "and", "or", "of", "&", "'n",
    "with", "without", "plus",
    "in", "on", "for", "to", "from",
    "the", "a", "an",

    # container / format
    "sauce", "juice", "syrup", "soup", "broth", "dressing", "topping", "spray",

    # bready / pastry forms
    "bread", "pie", "pasta", "noodle", "roll", "cake", "crust", "shell", "pastry", "dough",

    # “junk drawer” / vague
    "mixed", "seasoning", "spice", "concentrate", "drink",

    # generic candy / drinks
    "candy", "cookie", "cooky", "cracker", "cereal", "soda", "liqueur", "tea", "coffee",

    # generic veg/fruit umbrella terms
    "vegetable", "greens", "fruit",

    "powder", "seed", "salad", "bar", "round", "liquid", "food", "bag", "stick", "crescent", "unbaked", "9-inch", "extract", "water",
    "french", "english", "de", "chop", "italian", "chinese", "deep", "roasted", "dripping", "double-crust", "mexican", "sea", "greek",
    "cube", "7-up", "essence", "mix", "roast", "sausage", "original", "bit", "meat", "nut", "red", "blue", "spicy", "paper", "foil", 
    "wood", "wrapper", "toothpick", "bay", "top", "bottom", "dinner", "mountain", "paper", "cup", "wax", "wrap", "stick", "wooden",
    "toothpick", "pan", "spray", "crystal", "jar", "bottled", "slice", "puff", "creme", "pudding", "seasoned", "dish", "taco", "skewer",
    "bun", "toast", "smoke", "link", "whip", "whipped", "spiral", "flower", "cocktail", "preserve", "herb", "barbecue", "ice", "ranch",
    "tail", "cane", "chily", "half-and-half",
}

def choose_group_token(ingredient, token_freqs, stop_tokens):
    """
    Choose a single 'head' token for an ingredient string, based on:
      - splitting on spaces,
      - ignoring grouping stopwords,
      - ignoring tokens with no alphabetic characters,
      - picking the token with the highest overall frequency.

    Returns:
        A token string (e.g. "egg", "beef") or None if no suitable token.
    """
    tokens = ingredient.split()

    candidate_tokens = []
    for t in tokens:
        t_clean = t.strip()

        # Skip empty tokens
        if not t_clean:
            continue

        # Skip tokens with no letters (e.g., "95%", "1/2")
        if not any(c.isalpha() for c in t_clean):
            continue

        # Skip "glue" / function-word tokens that shouldn't be group heads
        if t_clean in stop_tokens:
            continue

        candidate_tokens.append(t_clean)

    if not candidate_tokens:
        # No usable token → no group head
        return None

    # Pick the candidate with the highest global token frequency
    head = max(candidate_tokens, key=lambda x: token_freqs.get(x, 0))

    return head


# 7.1 Build mapping: group_head -> list of ingredient strings
group_to_ingredients = defaultdict(list)
ingredient_to_group = {}

# Use the final vocabulary of ingredients
for ingredient in ingredient_vocab_df["ingredient"]:
    head_token = choose_group_token(ingredient, token_counts, GROUPING_STOP_TOKENS)

    # If we truly can't find any head token, fall back to the
    # full ingredient string itself as its own "group".
    if head_token is None:
        group_name = ingredient
    else:
        group_name = head_token

    group_to_ingredients[group_name].append(ingredient)
    ingredient_to_group[ingredient] = group_name


# 7.2 Turn groups into a DataFrame and display all of them
group_rows = []
for group_name, members in group_to_ingredients.items():
    # Sort members just for readability
    sorted_members = sorted(members)

    group_rows.append({
        "group_head": group_name,
        "group_size": len(sorted_members),
        "ingredients": ", ".join(sorted_members),
    })

groups_df = (
    pd.DataFrame(group_rows)
    .sort_values(by="group_size", ascending=False)
    .reset_index(drop=True)
)

# Only keep groups with more than 1 ingredient
multi_member_groups = groups_df[groups_df["group_size"] > 1]

print(f"Number of groups with size > 1: {len(multi_member_groups)}")

pd.set_option("display.max_rows", len(multi_member_groups))
display(multi_member_groups)



Number of groups with size > 1: 296


,group_head,group_size,ingredients
0,cheese,105,"1 fat cottage cheese, 2 cheddar cheese, 2 fat ..."
1,chocolate,55,"baker's chocolate, bittersweet chocolate, cand..."
2,beef,53,"80 beef, 85 beef, 90 beef, 95 beef, beef, beef..."
3,chicken,50,"broiler chicken, broiler-fryer chicken, chicke..."
4,flour,45,"10-inch flour tortilla, 6-inch flour tortilla,..."
5,cream,45,"18 table cream, 35 cream, 98 cream of chicken ..."
6,bean,42,"adzuki bean, baked bean, bean, bean dip, bean ..."
7,pepper,42,"banana pepper, bottled pepper sauce, cayenne p..."
8,tomato,40,"beefsteak tomato, campbell's tomato soup, cher..."
9,pork,29,"barbecued pork, bulk pork sausage, center-cut ..."


## Runtime Feasibility Check for Full Dataset

Before running the full trigger-detection pipeline, we first estimate whether the dataset is computationally manageable on a standard laptop.  
The goal of this section is to measure the time needed for:

1. building the ingredient co-occurrence graph  
2. approximating the cost of Node2Vec embedding  
3. confirming whether we can safely use the entire dataset  

We do this by benchmarking graph construction on small random subsets (such as 5k and 20k recipes) and then scaling the time linearly.  
This method works because co-occurrence graph construction is proportional to the number of recipes.

Node2Vec time is also estimated based on the total number of walk tokens:

number_of_nodes * walks_per_node * walk_length

This gives a simple and accurate approximation of how much work Word2Vec will need to do.  
These tests together show how much data we can realistically process without exceeding our runtime budget.


In [60]:
# -----------------------------------------------------------
# Final dataset summary + graph construction feasibility
# (lightweight version, no extra libraries)
# -----------------------------------------------------------
import time
from itertools import combinations

# 1) Basic stats about the final cleaned dataset
n_recipes = len(df)
ingredient_lengths = df["ingredients"].apply(len)

print("=== Final cleaned dataset summary ===")
print(f"Number of recipes: {n_recipes}")
print(f"Min ingredients per recipe: {ingredient_lengths.min()}")
print(f"Max ingredients per recipe: {ingredient_lengths.max()}")
print(f"Mean ingredients per recipe: {ingredient_lengths.mean():.2f}")
print(f"Median ingredients per recipe: {ingredient_lengths.median():.0f}")

# If you have ingredient_vocab_df from earlier, report vocab size as well
if "ingredient_vocab_df" in globals():
    print(f"Final ingredient vocabulary size: {len(ingredient_vocab_df)}")
print()

# 2) Helper: build co-occurrence edges on a subset
from collections import Counter

def build_cooccurrence_edges(df_subset):
    """
    Simple co-occurrence graph construction on a subset of recipes.
    Returns a Counter mapping (ingredient_a, ingredient_b) -> count.
    """
    edge_counts = Counter()
    for ingredients in df_subset["ingredients"]:
        # Work with unique ingredients per recipe
        unique_ings = sorted(set(ingredients))
        for a, b in combinations(unique_ings, 2):
            edge_counts[(a, b)] += 1
    return edge_counts

# 3) Runtime benchmarks on small subsets (fast, for feasibility)
subset_sizes = [5_000, 20_000]

print("=== Graph construction benchmarks (co-occurrence only) ===")
for subset_size in subset_sizes:
    if n_recipes < subset_size:
        continue  # skip if dataset is smaller than this subset size

    subset = df.sample(n=subset_size, random_state=42)

    start = time.perf_counter()
    edges = build_cooccurrence_edges(subset)
    elapsed = time.perf_counter() - start

    print(f"\n--- {subset_size} recipes ---")
    print(f"Unique edges in subset: {len(edges)}")
    print(f"Time taken on this machine: {elapsed:.2f} seconds")

    # Rough linear estimate for full dataset (graph construction only)
    est_full_seconds = elapsed * (n_recipes / subset_size)
    print(f"Estimated graph build time for full dataset: "
          f"{est_full_seconds/60:.2f} minutes")


=== Final cleaned dataset summary ===
Number of recipes: 219890
Min ingredients per recipe: 3
Max ingredients per recipe: 39
Mean ingredients per recipe: 6.93
Median ingredients per recipe: 6
Final ingredient vocabulary size: 2698

=== Graph construction benchmarks (co-occurrence only) ===

--- 5000 recipes ---
Unique edges in subset: 50980
Time taken on this machine: 0.04 seconds
Estimated graph build time for full dataset: 0.03 minutes

--- 20000 recipes ---
Unique edges in subset: 121352
Time taken on this machine: 0.13 seconds
Estimated graph build time for full dataset: 0.02 minutes


## Feasibility Results and Interpretation

The cleaning pipeline produced a final dataset of about 220k recipes with an average of roughly seven ingredients each.  
The final ingredient vocabulary contains about 2700 unique items, meaning the eventual co-occurrence graph will have 2700 nodes.

### Graph Construction Results

The benchmark shows:

- 5000 recipes -> about 0.05 seconds  
- 20000 recipes -> about 0.13 seconds  
- full dataset -> less than 0.1 minutes  

This confirms that graph construction is extremely fast and does not limit the system.

### Node2Vec Cost Estimation

Node2Vec runtime depends mainly on the amount of walk data it creates.  
We can estimate this using the simple multiplication:

number_of_nodes * walks_per_node * walk_length

For example, if we choose:

- number_of_nodes = 2700  
- walks_per_node = 10  
- walk_length = 40  

then:

2700 * 10 * 40  
= 1,080,000 walk tokens  

This is just over one million tokens.  
Training Word2Vec on one million tokens typically takes well under two minutes on a CPU.

If we double the number of walks:

2700 * 20 * 40  
= 2,160,000 tokens  

This is still only about two million tokens, which is also said to be very small.

### Overall Conclusion

- graph construction is effectively instant  
- Node2Vec walk generation and embedding training comfortably fit within the five-minute constraint  
- the full dataset (all 220k recipes) is computationally feasible  
- no downsampling is required  

These results confirm that the project can safely use the entire cleaned dataset for graph-based modelling and Node2Vec embeddings.

In [61]:
# -----------------------------------------------------------
# Step 7 – Fragmentation sanity check for core ingredients
# -----------------------------------------------------------
# At this point, the cleaning pipeline has produced a final ingredient
# vocabulary (ingredient_vocab_df). Before building any models on top
# of this, we run a quick "fragmentation check" on a few very common
# ingredients (milk, cheese, chicken, etc.).
#
# Goal:
#   - Detect whether one real-world ingredient has turned into many
#     slightly different strings (formatting variants).
#   - If fragmentation is low, we move on confidently.
#   - If fragmentation is high, we can add a small targeted rule later.
#
# This is a diagnostic-only step: it does NOT modify the dataset.

# -----------------------------------------------------------
# 7.1 Define a small set of common "base ingredients" to inspect
# -----------------------------------------------------------
# These should be ingredients that appear in many recipes and
# are expected to be very stable across naming.
CORE_INGREDIENT_QUERIES = [
    "milk",
    "cheese",
    "chicken",
    "tomato",
    "onion",
    "egg",
    "butter",
    "flour",
]

# -----------------------------------------------------------
# 7.2 Helper: find all vocabulary items that contain a query word
# -----------------------------------------------------------
def find_variants_in_vocab(query_word, vocab_series):
    """
    Return all ingredient strings in the vocabulary that contain
    the query word as a substring.

    Parameters
    ----------
    query_word : str
        A simple keyword such as "milk" or "cheese".
    vocab_series : pd.Series
        The ingredient column from ingredient_vocab_df.

    Returns
    -------
    variants : list[str]
        Sorted list of ingredient strings that contain the query.
    """
    q = query_word.strip().lower()
    variants = sorted([ing for ing in vocab_series if q in ing])
    return variants


# -----------------------------------------------------------
# 7.3 Run the check and print results in a readable format
# -----------------------------------------------------------
print("=== Fragmentation sanity check (core ingredients) ===")

# Ensure we have the vocabulary available before running the check.
if "ingredient_vocab_df" not in globals():
    raise ValueError("ingredient_vocab_df not found. Run the cleaning pipeline first.")

vocab_ings = ingredient_vocab_df["ingredient"]

for q in CORE_INGREDIENT_QUERIES:
    variants = find_variants_in_vocab(q, vocab_ings)

    print("\n--------------------------------------------------")
    print(f"Query: '{q}'")
    print(f"Number of variants found: {len(variants)}")

    # Print only the first N variants to avoid flooding the notebook output.
    # You can increase N if you want to inspect more.
    N = 80
    if len(variants) <= N:
        for v in variants:
            print(" -", v)
    else:
        for v in variants[:N]:
            print(" -", v)
        print(f"... (showing first {N} of {len(variants)})")

print("\n=== End of fragmentation check ===")

# -----------------------------------------------------------
# 7.4 (Optional) Quick frequency view for one query word
# -----------------------------------------------------------
# If you want to see which variants are most common, this block
# prints a small frequency table for one chosen word.
#
# Change INSPECT_WORD to any term in CORE_INGREDIENT_QUERIES.
INSPECT_WORD = "milk"

variants = find_variants_in_vocab(INSPECT_WORD, vocab_ings)

# Build a small DataFrame of counts for the variants, sorted by count.
variant_counts_df = (
    ingredient_vocab_df[ingredient_vocab_df["ingredient"].isin(variants)]
    .sort_values(by="count", ascending=False)
    .reset_index(drop=True)
)

print("\n=== Variant frequency table (optional) ===")
print(f"Inspect word: '{INSPECT_WORD}'")
display(variant_counts_df.head(30))


=== Fragmentation sanity check (core ingredients) ===

--------------------------------------------------
Query: 'milk'
Number of variants found: 29
 - 1 fat buttermilk
 - 1 milk
 - 2 milk
 - almond milk
 - buttermilk
 - buttermilk biscuit
 - buttermilk pancake
 - buttermilk ranch salad
 - carnation milk
 - chocolate milk
 - coconut milk
 - eagle brand milk
 - full-fat milk
 - malted milk ball
 - malted milk powder
 - milk
 - milk chocolate
 - milk chocolate candy bar
 - milk chocolate kisse
 - milk powder
 - milk ricotta cheese
 - milky way bar
 - non-fat milk
 - pillsbury grand buttermilk biscuit
 - ready-to-bake buttermilk flaky biscuit
 - rice milk
 - soymilk
 - vanilla soymilk
 - vanilla-flavored soymilk

--------------------------------------------------
Query: 'cheese'
Number of variants found: 107
 - 1 fat cottage cheese
 - 2 cheddar cheese
 - 2 fat cottage cheese
 - aged cheddar cheese
 - american cheese
 - asadero cheese
 - asiago cheese
 - bleu cheese salad
 - blue cheese
 -

,ingredient_id,ingredient,count
0,1519,milk,35664
1,353,buttermilk,4316
2,656,coconut milk,2074
3,2297,soymilk,746
4,1520,milk chocolate,729
5,7,2 milk,574
6,2,1 milk,460
7,1523,milk powder,444
8,354,buttermilk biscuit,226
9,62,almond milk,153


In [62]:
# -----------------------------------------------------------
# A.FINAL – Save the cleaned dataset + lookup tables to disk
# -----------------------------------------------------------
# This cell "freezes" the outputs of the cleaning pipeline so that:
#   1) we do NOT need to rerun cleaning every time,
#   2) downstream steps (graph building, embeddings, trigger detection) can load
#      stable, versioned files from /data/processed,
#   3) we can easily look up:
#        - recipe -> cleaned ingredients
#        - ingredient -> ingredient_id
#        - ingredient -> group_head
#
# Notes:
# - We save a small "recipe_ingredients" table for fast loading.
# - We also save the full cleaned recipes table (df) in case you want extra fields later.
# - We save ingredient vocabulary and ingredient->group mapping as lookup tables.

from pathlib import Path
import json

# -----------------------------------------------------------
# 1) Choose output folder
# -----------------------------------------------------------
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------
# 2) Identify the recipe id column (Food.com usually uses "id")
# -----------------------------------------------------------
RECIPE_ID_COL = "id"
if RECIPE_ID_COL not in df.columns:
    raise ValueError(
        f"Expected a recipe id column named '{RECIPE_ID_COL}', "
        f"but it was not found. Available columns: {list(df.columns)[:20]} ..."
    )

# -----------------------------------------------------------
# 3) Build + save a minimal "recipe -> ingredients" table
# -----------------------------------------------------------
# This is the main table you will load later for:
#   - graph construction
#   - trigger detection
#   - recipe lookup
#
# We keep ingredients_raw for debugging and reporting.
# We keep n_clean_ingredients because it is useful for sanity checks.
cols_to_save = [RECIPE_ID_COL, "ingredients", "ingredients_raw"]
if "n_clean_ingredients" in df.columns:
    cols_to_save.append("n_clean_ingredients")

recipe_ingredients_df = df[cols_to_save].copy()

recipe_ingredients_path = OUT_DIR / "recipe_ingredients.csv"
recipe_ingredients_df.to_csv(recipe_ingredients_path, index=False)

print("Saved:", recipe_ingredients_path)
print("Rows:", len(recipe_ingredients_df), "| Columns:", list(recipe_ingredients_df.columns))

# -----------------------------------------------------------
# 4) Save the full cleaned recipes table (optional but useful)
# -----------------------------------------------------------
# This keeps all original recipe fields + cleaned ingredients.
# If disk space becomes a concern, you can comment this out.
recipes_cleaned_path = OUT_DIR / "recipes_cleaned.csv"
df.to_csv(recipes_cleaned_path, index=False)

print("Saved:", recipes_cleaned_path)
print("Rows:", len(df), "| Columns:", df.shape[1])

# -----------------------------------------------------------
# 5) Save the ingredient vocabulary (ingredient -> ingredient_id -> count)
# -----------------------------------------------------------
# This is needed for stable ids and later matrix/graph building.
ingredient_vocab_path = OUT_DIR / "ingredient_vocabulary.csv"
ingredient_vocab_df.to_csv(ingredient_vocab_path, index=False)

print("Saved:", ingredient_vocab_path)
print("Vocab size:", len(ingredient_vocab_df))

# -----------------------------------------------------------
# 6) Save the ingredient -> group mapping (lookup table)
# -----------------------------------------------------------
# This lets you look up which group (group_head) an ingredient belongs to.
# We save it in a clean 2-column format.
if "ingredient_to_group" not in globals():
    raise ValueError("ingredient_to_group not found. Run the grouping section first.")

ingredient_groups_df = pd.DataFrame(
    [{"ingredient": ing, "group_head": grp} for ing, grp in ingredient_to_group.items()]
).sort_values(by=["group_head", "ingredient"]).reset_index(drop=True)

ingredient_groups_path = OUT_DIR / "ingredient_groups.csv"
ingredient_groups_df.to_csv(ingredient_groups_path, index=False)

print("Saved:", ingredient_groups_path)
print("Rows:", len(ingredient_groups_df))

# -----------------------------------------------------------
# 7) (Optional) Save a small metadata JSON for reproducibility
# -----------------------------------------------------------
# This records key thresholds and outputs in one place.
metadata = {
    "recipe_id_col": RECIPE_ID_COL,
    "min_ingredient_frequency": MIN_INGREDIENT_FREQUENCY,
    "max_ingredient_frequency": MAX_INGREDIENT_FREQUENCY,
    "min_ingredients_per_recipe": MIN_INGREDIENTS_PER_RECIPE,
    "n_recipes_final": int(len(df)),
    "vocab_size_final": int(len(ingredient_vocab_df)),
    "files_written": {
        "recipe_ingredients_csv": str(recipe_ingredients_path),
        "recipes_cleaned_csv": str(recipes_cleaned_path),
        "ingredient_vocabulary_csv": str(ingredient_vocab_path),
        "ingredient_groups_csv": str(ingredient_groups_path),
    },
}

metadata_path = OUT_DIR / "cleaning_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", metadata_path)

print("\n=== Freeze complete ===")


Saved: ../data/processed/recipe_ingredients.csv
Rows: 219890 | Columns: ['id', 'ingredients', 'ingredients_raw', 'n_clean_ingredients']
Saved: ../data/processed/recipes_cleaned.csv
Rows: 219890 | Columns: 14
Saved: ../data/processed/ingredient_vocabulary.csv
Vocab size: 2698
Saved: ../data/processed/ingredient_groups.csv
Rows: 2698
Saved: ../data/processed/cleaning_metadata.json

=== Freeze complete ===


In [63]:
# -----------------------------------------------------------
# A.CHECK – Reload the frozen outputs and sanity check them
# -----------------------------------------------------------
# Goal:
# 1) Confirm files were written correctly
# 2) Confirm row matching is still correct via recipe id
# 3) Confirm ingredient/group lookups work from disk

from pathlib import Path
import pandas as pd
import ast

OUT_DIR = Path("../data/processed")

# Load frozen tables
recipe_ingredients = pd.read_csv(OUT_DIR / "recipe_ingredients.csv")
ingredient_vocab = pd.read_csv(OUT_DIR / "ingredient_vocabulary.csv")
ingredient_groups = pd.read_csv(OUT_DIR / "ingredient_groups.csv")

print("Loaded frozen tables:")
print(" - recipe_ingredients:", recipe_ingredients.shape)
print(" - ingredient_vocabulary:", ingredient_vocab.shape)
print(" - ingredient_groups:", ingredient_groups.shape)

# Convert ingredients column back into a real list (CSV stores it as a string)
recipe_ingredients["ingredients"] = recipe_ingredients["ingredients"].apply(ast.literal_eval)

# 1) Basic checks
assert recipe_ingredients["id"].isna().sum() == 0, "Found missing recipe ids."
assert recipe_ingredients["id"].duplicated().sum() == 0, "Found duplicate recipe ids."
assert ingredient_vocab["ingredient_id"].duplicated().sum() == 0, "Duplicate ingredient_id."
assert ingredient_vocab["ingredient"].duplicated().sum() == 0, "Duplicate ingredient strings in vocab."

print("\nBasic integrity checks passed.")

# 2) Check that every ingredient appearing in recipes exists in the vocabulary
all_recipe_ings = set(x for lst in recipe_ingredients["ingredients"] for x in lst)
vocab_ings = set(ingredient_vocab["ingredient"])
missing_in_vocab = sorted(list(all_recipe_ings - vocab_ings))

print("\nIngredient coverage check:")
print(" - unique ingredients in recipes:", len(all_recipe_ings))
print(" - unique ingredients in vocab:", len(vocab_ings))
print(" - missing in vocab:", len(missing_in_vocab))
if missing_in_vocab[:10]:
    print("   examples:", missing_in_vocab[:10])

# 3) Check group lookup works (ingredient -> group_head)
group_lookup = dict(zip(ingredient_groups["ingredient"], ingredient_groups["group_head"]))
example_ing = next(iter(all_recipe_ings))
print("\nExample group lookup:")
print(" - ingredient:", example_ing)
print(" - group_head:", group_lookup.get(example_ing, "<NOT FOUND>"))

print("\n=== Frozen data looks usable ===")


Loaded frozen tables:
 - recipe_ingredients: (219890, 4)
 - ingredient_vocabulary: (2698, 3)
 - ingredient_groups: (2698, 2)

Basic integrity checks passed.

Ingredient coverage check:
 - unique ingredients in recipes: 2698
 - unique ingredients in vocab: 2698
 - missing in vocab: 0

Example group lookup:
 - ingredient: dill pickle slice
 - group_head: dill

=== Frozen data looks usable ===
